# Week 6, Thursday

## Course project

Instructions for the course project are posted on Canvas.  Especially important, please keep things manageable and understandable (by you and by me), and use concepts from our course (and potentially 220A/290A).

There is a "Final Project Preparation" homework assignment due Friday of Week 8 (Nov 20).  Part of that assignment is meeting with Luke during his office hours for approximately 10 minutes to discuss your idea for the project or to ask questions / get ideas.  Plan to have that meeting sometime before the due date of that Preparation assignment.

## Evaluating a Neural Network by hand

* What is the difference between "evaluating" a Neural Network and "fitting" a Neural Network?

Here by evaluating, I mean "plug in some inputs, get some outputs".  By fitting, I mean attempt to find what coefficients work well.  (Fitting is done using a variant of gradient descent that we'll talk about later.)

* For the penguins dataset from sckikit-learn, fit a neural network classifier with a single hidden layer of 4 neurons using "bill_length_mm" and "bill_depth_mm" as predictors, and "species" as target.  Use ReLU as our hidden layer activation function, and use "lbfgs" as our solver (my understanding is that this solver is a good choice for relatively small datasets).  Use a random state of 0 and a maximum of 1000 iterations.

In [1]:
import seaborn as sns
import numpy as np
from sklearn.neural_network import MLPClassifier

np.set_printoptions(suppress=True, precision=4) # remove exponential notation

In [2]:
df = sns.load_dataset("penguins")
df = df.dropna(axis=0).copy()

In [3]:
# We convert to NumPy arrays to make rescaling for softmax a little more intuitive
X = df[["bill_length_mm", "bill_depth_mm"]].to_numpy()
y = df["species"].to_numpy()

In [4]:
clf = MLPClassifier(
    hidden_layer_sizes=(4,),
    activation="relu",
    solver="lbfgs",
    random_state=0,
    max_iter=1000
)

In [5]:
clf.fit(X, y)

,"hidden_layer_sizes hidden_layer_sizes: array-like of shape(n_layers - 2,), default=(100,)The ith element represents the number of neurons in the ithhidden layer.","(4,)"
,"solver solver: {'lbfgs', 'sgd', 'adam'}, default='adam'The solver for weight optimization.- 'lbfgs' is an optimizer in the family of quasi-Newton methods.- 'sgd' refers to stochastic gradient descent.- 'adam' refers to a stochastic gradient-based optimizer proposed by Kingma, Diederik, and Jimmy BaFor a comparison between Adam optimizer and SGD, see:ref:`sphx_glr_auto_examples_neural_networks_plot_mlp_training_curves.py`.Note: The default solver 'adam' works pretty well on relativelylarge datasets (with thousands of training samples or more) in terms ofboth training time and validation score.For small datasets, however, 'lbfgs' can converge faster and performbetter.",'lbfgs'
,"max_iter max_iter: int, default=200Maximum number of iterations. The solver iterates until convergence(determined by 'tol') or this number of iterations. For stochasticsolvers ('sgd', 'adam'), note that this determines the number of epochs(how many times each data point will be used), not the number ofgradient steps.",1000
,"random_state random_state: int, RandomState instance, default=NoneDetermines random number generation for weights and biasinitialization, train-test split if early stopping is used, and batchsampling when solver='sgd' or 'adam'.Pass an int for reproducible results across multiple function calls.See :term:`Glossary <random_state>`.",0
,"activation activation: {'identity', 'logistic', 'tanh', 'relu'}, default='relu'Activation function for the hidden layer.- 'identity', no-op activation, useful to implement linear bottleneck, returns f(x) = x- 'logistic', the logistic sigmoid function, returns f(x) = 1 / (1 + exp(-x)).- 'tanh', the hyperbolic tan function, returns f(x) = tanh(x).- 'relu', the rectified linear unit function, returns f(x) = max(0, x)",'relu'
,"alpha alpha: float, default=0.0001Strength of the L2 regularization term. The L2 regularization termis divided by the sample size when added to the loss.For an example usage and visualization of varying regularization, see:ref:`sphx_glr_auto_examples_neural_networks_plot_mlp_alpha.py`.",0.0001
,"batch_size batch_size: int, default='auto'Size of minibatches for stochastic optimizers.If the solver is 'lbfgs', the classifier will not use minibatch.When set to ""auto"", `batch_size=min(200, n_samples)`.",'auto'
,"learning_rate learning_rate: {'constant', 'invscaling', 'adaptive'}, default='constant'Learning rate schedule for weight updates.- 'constant' is a constant learning rate given by 'learning_rate_init'.- 'invscaling' gradually decreases the learning rate at each time step 't' using an inverse scaling exponent of 'power_t'. effective_learning_rate = learning_rate_init / pow(t, power_t)- 'adaptive' keeps the learning rate constant to 'learning_rate_init' as long as training loss keeps decreasing. Each time two consecutive epochs fail to decrease training loss by at least tol, or fail to increase validation score by at least tol if 'early_stopping' is on, the current learning rate is divided by 5.Only used when ``solver='sgd'``.",'constant'
,"learning_rate_init learning_rate_init: float, default=0.001The initial learning rate used. It controls the step-sizein updating the weights. Only used when solver='sgd' or 'adam'.",0.001
,"power_t power_t: float, default=0.5The exponent for inverse scaling learning rate.It is used in updating effective learning rate when the learning_rateis set to 'invscaling'. Only used when solver='sgd'.",0.5
,"shuffle shuffle: bool, default=TrueWhether to shuffle samples in each iteration. Only used whensolver='sgd' or 'adam'.",True


* Try to build up the outputs of this neural network step by step, using the learned coefficients and intercepts.

In [6]:
# which column which species?
clf.classes_

array(['Adelie', 'Chinstrap', 'Gentoo'], dtype='<U9')

In [7]:
X0 = X.copy()

In [8]:
w0 = clf.coefs_[0]
b0 = clf.intercepts_[0]

In [9]:
# define a relu function

def relu(x):
    return np.maximum(0, x)

In [10]:
X1 = relu(X0@w0 + b0)

In [11]:
w1 = clf.coefs_[1]
b1 = clf.intercepts_[1]

In [13]:
w1.shape

(4, 3)

In [14]:
b1.shape

(3,)

In [12]:
# scores, don't apply activation to the output later
Z = X1@w1 + b1

In [15]:
Z[:5]

array([[34.7511, 21.7988,  8.2763],
       [29.4875, 24.0427, 14.4101],
       [31.0374, 24.1883, 13.6779],
       [39.1592, 18.1054,  0.723 ],
       [41.7432, 19.4604,  0.984 ]])

Z is sometimes called a score or log odds.  We need to turn it into probabilities.  We will use a hand-defined softmax function (no sigmoid), where $z_i$ gets replaced by $e^{z_i}/\sum_j e^{z_j}$.

The key properties of this strange-looking softmax function are the following.  Notice how these are natural if we want to turn the score vector into a probability vector:
- Preserves the ordering (i.e., is monotone increasing);
- Produces numbers between 0 and 1;
- Produces numbers that sum to 1.

In [17]:
np.exp(Z)[:5]

array([[1.2365e+15, 2.9314e+09, 3.9297e+03],
       [6.4012e+12, 2.7645e+10, 1.8123e+06],
       [3.0156e+13, 3.1976e+10, 8.7148e+05],
       [1.0154e+17, 7.2955e+07, 2.0606e+00],
       [1.3453e+18, 2.8285e+08, 2.6752e+00]])

In [18]:
np.exp(Z)[:5].sum(axis=1)

array([1.2365e+15, 6.4289e+12, 3.0188e+13, 1.0154e+17, 1.3453e+18])

In [19]:
# right spiritually but not in NumPy syntax
np.exp(Z) / np.exp(Z).sum(axis=1)

ValueError: operands could not be broadcast together with shapes (333,3) (333,) 

In [21]:
Z + np.asarray([[0,1,2],[1,2,3]])

ValueError: operands could not be broadcast together with shapes (333,3) (2,3) 

In [ ]:
# our probabilities, output of softmax
np.exp(Z) / np.exp(Z).sum(axis=1).reshape(-1,1)

array([[1.    , 0.    , 0.    ],
       [0.9957, 0.0043, 0.    ],
       [0.9989, 0.0011, 0.    ],
       [1.    , 0.    , 0.    ],
       [1.    , 0.    , 0.    ],
       [0.9998, 0.0002, 0.    ],
       [1.    , 0.    , 0.    ],
       [0.9578, 0.0422, 0.    ],
       [1.    , 0.    , 0.    ],
       [1.    , 0.    , 0.    ],
       [1.    , 0.    , 0.    ],
       [1.    , 0.    , 0.    ],
       [1.    , 0.    , 0.    ],
       [1.    , 0.    , 0.    ],
       [1.    , 0.    , 0.    ],
       [1.    , 0.    , 0.    ],
       [1.    , 0.    , 0.    ],
       [1.    , 0.    , 0.    ],
       [1.    , 0.    , 0.    ],
       [0.9972, 0.0028, 0.    ],
       [1.    , 0.    , 0.    ],
       [0.9999, 0.0001, 0.    ],
       [0.9973, 0.0027, 0.    ],
       [1.    , 0.    , 0.    ],
       [1.    , 0.    , 0.    ],
       [0.8644, 0.1355, 0.0001],
       [1.    , 0.    , 0.    ],
       [0.9994, 0.0006, 0.    ],
       [1.    , 0.    , 0.    ],
       [0.9999, 0.0001, 0.    ],
       [1.

In [25]:
# another way
probs = np.exp(Z) / np.exp(Z).sum(axis=1, keepdims=True)
probs[:5]

array([[1.    , 0.    , 0.    ],
       [0.9957, 0.0043, 0.    ],
       [0.9989, 0.0011, 0.    ],
       [1.    , 0.    , 0.    ],
       [1.    , 0.    , 0.    ]])

In [24]:
np.exp(Z).sum(axis=1, keepdims=True)

array([[1.2365e+15],
       [6.4289e+12],
       [3.0188e+13],
       [1.0154e+17],
       [1.3453e+18],
       [4.9511e+13],
       [3.3772e+16],
       [3.4052e+12],
       [2.4199e+19],
       [5.9092e+20],
       [3.8621e+14],
       [5.4838e+15],
       [1.1257e+17],
       [2.6525e+16],
       [1.0115e+17],
       [8.7269e+14],
       [4.3179e+15],
       [1.4225e+17],
       [2.8701e+14],
       [5.6389e+12],
       [7.8356e+16],
       [2.2206e+14],
       [1.7340e+13],
       [2.4762e+15],
       [7.5326e+14],
       [5.2746e+11],
       [7.0113e+14],
       [2.8982e+13],
       [5.2697e+14],
       [2.2552e+13],
       [9.7083e+18],
       [2.1845e+17],
       [3.6623e+13],
       [4.5446e+16],
       [2.9942e+15],
       [8.9831e+14],
       [8.7338e+13],
       [9.2664e+15],
       [6.1932e+14],
       [9.0499e+12],
       [1.1539e+15],
       [6.4285e+14],
       [9.6267e+14],
       [8.8823e+17],
       [1.8184e+13],
       [1.0767e+15],
       [2.3517e+15],
       [1.898

In [26]:
clf.predict_proba(X)[:5]

array([[1.    , 0.    , 0.    ],
       [0.9957, 0.0043, 0.    ],
       [0.9989, 0.0011, 0.    ],
       [1.    , 0.    , 0.    ],
       [1.    , 0.    , 0.    ]])

In [27]:
# spiritually correct, but problems with this
# numerical precision issues
# and will be many Trues/Falses as opposed to 1
clf.predict_proba(X) == (np.exp(Z) / np.exp(Z).sum(axis=1, keepdims=True))

array([[False,  True, False],
       [False, False,  True],
       [False, False,  True],
       [ True,  True, False],
       [False,  True, False],
       [ True,  True, False],
       [False,  True, False],
       [ True, False,  True],
       [False, False, False],
       [ True, False, False],
       [ True, False, False],
       [False, False, False],
       [False, False, False],
       [ True,  True, False],
       [False,  True, False],
       [False, False, False],
       [False, False, False],
       [ True, False, False],
       [False, False,  True],
       [ True, False,  True],
       [ True,  True, False],
       [False,  True, False],
       [False,  True, False],
       [False, False, False],
       [False, False, False],
       [ True,  True,  True],
       [False, False, False],
       [ True,  True, False],
       [ True, False, False],
       [ True, False, False],
       [ True,  True,  True],
       [False, False, False],
       [ True, False, False],
       [Fa

In [29]:
np.allclose(clf.predict_proba(X), (np.exp(Z) / np.exp(Z).sum(axis=1, keepdims=True)))

True

## Log loss

We can compute the log loss for our probability estimates using the following.

Reminders: log loss is more refined than accuracy.  Log loss has nothing to do with neural networks, works equally well for any classification problem.

In [30]:
from sklearn.metrics import log_loss

In [31]:
log_loss(y, clf.predict_proba(X), labels=clf.classes_)

0.1177449320917818

In [35]:
y[251]

'Gentoo'

In [36]:
clf.predict_proba(X)[251]

array([0.    , 0.0078, 0.9922])

In [37]:
clf.classes_

array(['Adelie', 'Chinstrap', 'Gentoo'], dtype='<U9')

In [38]:
-np.log(0.9922)

np.float64(0.007830579115188564)

In [39]:
clf.predict_proba(X)[250]

array([0.0048, 0.987 , 0.0082])

In [40]:
-np.log(0.0082)

np.float64(4.803621124711929)

* Can we reproduce the that number from the probabilities directly?  The formula for log loss is $\frac{1}{n} \sum -\log(p_i)$, where $n$ is the number of observations, and where $p_i$ is the predicted probability assigned to the true value of $y_i$.  (As an example of what needs to be done, consider the second-to-last row.)

In [41]:
probs[-2]

array([0.    , 0.0622, 0.9378])

In [42]:
y[-2]

'Gentoo'

In [44]:
clf.classes_

array(['Adelie', 'Chinstrap', 'Gentoo'], dtype='<U9')

In [45]:
clf.classes_.index("Adelie")

AttributeError: 'numpy.ndarray' object has no attribute 'index'

In [46]:
class_list = list(clf.classes_)

In [47]:
class_list.index("Adelie")

0

In [48]:
class_list.index("Gentoo")

2

In [50]:
# which column to take for each row?
p_list = []
for label, p_row in zip(y, probs):
    ind = class_list.index(label)
    p_list.append(p_row[ind])

In [51]:
len(p_list)

333

In [54]:
(-np.log(p_list)).mean()

np.float64(0.1177449320917818)

## Review

Some topics.  Use lectures, homework, and sample midterm to get a sense of topics that might appear.

* Data types, with advantages, disadvantages

* Indexing (especially Boolean indexing)

* Plotting with Matplotlib and Seaborn

* Random numbers and probability estimates

* Linear regression and predictions

* Logistic regression and predictions, sigmoid

* Loss functions (including log loss)

* Overfitting

* Bias variance decomposition

* Neural networks and predictions